# 🎓 EX50: Hyperparameter Tuning for YOLO

Welcome, class! Today, we will explore the fascinating world of **Hyperparameter Tuning** for YOLO object detection models.

Unlike model parameters (weights and biases), which are learned automatically via backpropagation during training, **hyperparameters** are the architectural and optimization settings that we must configure beforehand. Finding the right mix is crucial for maximizing model performance, especially when training on specialized datasets.

---

## 🔍 Key YOLO Hyperparameters

To tune a YOLO model effectively, we must understand the levers we can pull:
1. **`lr0` (Initial Learning Rate):** Determines the step size at the start of training. If too high, weights will oscillate or explode; if too low, training will be slow or get stuck in local minima.
2. **`lrf` (Final Learning Rate Fraction):** The final learning rate is calculated as `lr0 * lrf`. This determines how much the learning rate shrinks by the end of training.
3. **`momentum` (Gradient Momentum):** Acceleration factor for weight updates to stabilize gradients (typically set to `0.937`).
4. **`weight_decay` (L2 Regularization):** Penalty to prevent weights from growing excessively large and reduce overfitting.
5. **`mosaic` (Mosaic Augmentation):** Combines 4 training images into one. This forces YOLO to learn object details at different scales and reduces reliance on global image context.

## 📉 Cosine Annealing Scheduler Math

YOLO decays the learning rate $\eta_t$ at epoch $t$ using a **Cosine Annealing** schedule:

$$\eta_t = \eta_{\min} + \frac{1}{2}(\eta_{\max} - \eta_{\min})\left(1 + \cos\left(rac{T_{cur}}{T_{\max}}\pi\right)\right)$$

Where:
* $\eta_{\max}$ is the initial learning rate (`lr0`).
* $\eta_{\min}$ is the target final learning rate (`lr0 * lrf`).
* $T_{cur}$ is the current epoch index (0-indexed).
* $T_{\max}$ is the total number of epochs.

### Why Cosine Annealing?
At the start of training, we want a high learning rate to quickly find good basins in the loss landscape. As training progresses, we want to smoothly reduce the learning rate to make finer adjustments. The cosine curve matches this intuition perfectly: it starts with a slow decay, accelerates in the middle, and flattens out towards the end.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

def get_cosine_lr(epoch: int, total_epochs: int, lr0: float, lrf: float) -> float:
    t_cur = min(epoch, total_epochs)
    eta_max = lr0
    eta_min = lr0 * lrf
    cos_factor = math.cos((t_cur / total_epochs) * math.pi)
    return eta_min + 0.5 * (eta_max - eta_min) * (1.0 + cos_factor)

# Parameters
lr0 = 0.01
lrf = 0.01
total_epochs = 100

epochs = np.arange(total_epochs + 1)
lrs = [get_cosine_lr(e, total_epochs, lr0, lrf) for e in epochs]

plt.figure(figsize=(10, 5))
plt.plot(epochs, lrs, label='Cosine Annealing', color='teal', linewidth=2)
plt.axhline(y=lr0, color='r', linestyle='--', label=f'Initial LR (lr0 = {lr0})')
plt.axhline(y=lr0 * lrf, color='g', linestyle='--', label=f'Final LR (lr0*lrf = {lr0*lrf})')
plt.title('Cosine Annealing Learning Rate Decay Curve', fontsize=14)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Learning Rate', fontsize=12)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(fontsize=11)
plt.show()

## 🧬 Genetic Algorithms for Hyperparameter Search

A **Genetic Algorithm (GA)** mimics natural selection to automatically search for optimal hyperparameters:
1. **Initialize:** Start with a base set of hyperparameters (the "parent").
2. **Mutate:** Create offspring by adding random noise to the parent's parameters. We sample from a Normal distribution:
   $$\Delta \sim N(0, \sigma \cdot \text{range})$$
   where $\text{range} = \text{upper\_bound} - \text{lower\_bound}$ and $\sigma$ is the mutation strength.
3. **Clip:** Ensure the mutated values remain within physically meaningful boundaries (e.g., momentum must be between 0.6 and 0.999; weight_decay >= 0.0).
4. **Evaluate:** Train a model with the mutated parameters for a few epochs and record its fitness (e.g., a combination of mAP@0.5 and mAP@0.5:0.95).
5. **Select:** The best-performing offspring becomes the parent for the next generation.

In [ ]:
import random
from typing import Dict, Tuple

def mutate_hyperparameters(
    parent_hyp: Dict[str, float],
    bounds: Dict[str, Tuple[float, float]],
    mutation_rate: float = 0.8,
    sigma: float = 0.1
) -> Dict[str, float]:
    mutated_hyp = {}
    for key, parent_val in parent_hyp.items():
        if key not in bounds:
            mutated_hyp[key] = parent_val
            continue
        lower, upper = bounds[key]
        param_range = upper - lower
        if random.random() < mutation_rate:
            mutation = random.gauss(0, sigma * param_range)
            mutated_val = parent_val + mutation
            clipped_val = max(lower, min(upper, mutated_val))
            mutated_hyp[key] = clipped_val
            assert lower <= clipped_val <= upper, "Out of bounds!"
        else:
            mutated_hyp[key] = parent_val
    return mutated_hyp

# Base/parent hyperparameters
parent_hyp = {'momentum': 0.937}
bounds = {'momentum': (0.6, 0.999)}

# Generate 10,000 mutations to visualize distribution
random.seed(42)
mutations = [mutate_hyperparameters(parent_hyp, bounds, mutation_rate=1.0, sigma=0.15)['momentum'] for _ in range(10000)]

plt.figure(figsize=(10, 5))
plt.hist(mutations, bins=50, color='darkslateblue', edgecolor='white', alpha=0.8)
plt.axvline(x=parent_hyp['momentum'], color='r', linestyle='--', label=f"Parent Momentum ({parent_hyp['momentum']})")
plt.axvline(x=bounds['momentum'][0], color='orange', linestyle=':', label='Lower Bound (0.6)')
plt.axvline(x=bounds['momentum'][1], color='orange', linestyle=':', label='Upper Bound (0.999)')
plt.title('Distribution of Mutated Momentum Hyperparameter (with Clipping)', fontsize=14)
plt.xlabel('Mutated Value', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

## 🔬 Case Study: Tuning YOLO for PTT Industrial Objects

Let's apply our knowledge to a practical example: training YOLO on the `overall-ptt-object-detection.v11i.yolov11` dataset. This dataset features thin and small objects such as `small-valve`, `lever-handle`, and `pig-alert`.

### Educator Recommendations:
1. **Reduce Spatial Scale Augmentation (`scale`):** By default, YOLO randomly scales images up and down (e.g., `scale=0.5` allows reducing an image to half its size). For tiny valves or levers, scaling them down makes them disappear entirely from the feature maps. We should mutate `scale` to be smaller (e.g., bounds `(0.1, 0.3)` instead of `(0.0, 0.9)`) to keep spatial features intact.
2. **Tune Mosaic Closing Epochs (`close_mosaic`):** Mosaic augmentation combines 4 training images into one. While this is great for teaching the model context-free feature learning in early stages, the boundaries of combined images can confuse the localization loss towards the end of training. Setting `close_mosaic` (e.g., closing it 15-20 epochs before the end) lets the model fine-tune on clean, un-distorted object boundaries, significantly boosting final localization accuracy (mAP@0.5:0.95).

---
*Related Topics:*
* [[EX49_YOLO_Result_Analysis]] - Analysing training results and loss curves.
* [[EX27_Learning_Rate]] - Understanding learning rate fundamentals.
* Return to main plan: [[YOLO_Learning_Plan]]
* Progress log: [[learning_journal]]